# 01 · Play-by-Play Exploration
Loads 2025 PBP via nflreadpy and audits shape, types, nulls, duplicates, and schema changes vs 2021.

In [1]:
import nflreadpy
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1 · Load 2025 play-by-play

In [2]:
pbp_2025 = nflreadpy.load_pbp([2025]).to_pandas()
print(f"Shape: {pbp_2025.shape}")

Shape: (48771, 372)


## 2 · Shape, dtypes, column list

In [3]:
print("\n--- dtypes ---")
print(pbp_2025.dtypes.to_string())


--- dtypes ---
play_id                                 float64
game_id                                     str
old_game_id                                 str
home_team                                   str
away_team                                   str
season_type                                 str
week                                      int32
posteam                                     str
posteam_type                                str
defteam                                     str
side_of_field                               str
yardline_100                            float64
game_date                                   str
quarter_seconds_remaining               float64
half_seconds_remaining                  float64
game_seconds_remaining                  float64
game_half                                   str
quarter_end                             float64
drive                                   float64
sp                                      float64
qtr                     

In [4]:
print("\n--- columns ---")
for i, col in enumerate(pbp_2025.columns):
    print(f"  [{i:03d}] {col}")


--- columns ---
  [000] play_id
  [001] game_id
  [002] old_game_id
  [003] home_team
  [004] away_team
  [005] season_type
  [006] week
  [007] posteam
  [008] posteam_type
  [009] defteam
  [010] side_of_field
  [011] yardline_100
  [012] game_date
  [013] quarter_seconds_remaining
  [014] half_seconds_remaining
  [015] game_seconds_remaining
  [016] game_half
  [017] quarter_end
  [018] drive
  [019] sp
  [020] qtr
  [021] down
  [022] goal_to_go
  [023] time
  [024] yrdln
  [025] ydstogo
  [026] ydsnet
  [027] desc
  [028] play_type
  [029] yards_gained
  [030] shotgun
  [031] no_huddle
  [032] qb_dropback
  [033] qb_kneel
  [034] qb_spike
  [035] qb_scramble
  [036] pass_length
  [037] pass_location
  [038] air_yards
  [039] yards_after_catch
  [040] run_location
  [041] run_gap
  [042] field_goal_result
  [043] kick_distance
  [044] extra_point_result
  [045] two_point_conv_result
  [046] home_timeouts_remaining
  [047] away_timeouts_remaining
  [048] timeout
  [049] timeout_tea

## 3 · Null audit on model-critical columns

In [5]:
# winner/result field name differs across nflreadpy versions; probe for it
result_candidates = [c for c in pbp_2025.columns
                     if 'result' in c.lower() or 'winner' in c.lower()]
print("Result/winner candidates:", result_candidates)

AUDIT_COLS = [
    'score_differential',
    'game_seconds_remaining',
    'yardline_100',
    'down',
    'ydstogo',
    'posteam_timeouts_remaining',
    'defteam_timeouts_remaining',
] + result_candidates

# Keep only columns that actually exist
present = [c for c in AUDIT_COLS if c in pbp_2025.columns]
missing = [c for c in AUDIT_COLS if c not in pbp_2025.columns]
if missing:
    print(f"\nNot found in dataset: {missing}")

null_counts = pbp_2025[present].isnull().sum()
null_pct    = (null_counts / len(pbp_2025) * 100).round(2)
null_report = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
print("\n--- Null audit ---")
print(null_report.to_string())

Result/winner candidates: ['field_goal_result', 'extra_point_result', 'two_point_conv_result', 'replay_or_challenge_result', 'series_result', 'fixed_drive_result', 'result']

--- Null audit ---
                            null_count  null_pct
score_differential                2729      5.60
game_seconds_remaining               4      0.01
yardline_100                      3548      7.27
down                              7993     16.39
ydstogo                              0      0.00
posteam_timeouts_remaining        2729      5.60
defteam_timeouts_remaining        2729      5.60
field_goal_result                47631     97.66
extra_point_result               47447     97.29
two_point_conv_result            48641     99.73
replay_or_challenge_result       48436     99.31
series_result                        1      0.00
fixed_drive_result                   1      0.00
result                               0      0.00


## 4 · Duplicate play_id check (within game)

In [6]:
dups = (
    pbp_2025
    .groupby(['game_id', 'play_id'])
    .size()
    .reset_index(name='count')
    .query('count > 1')
)
print(f"Duplicate (game_id, play_id) pairs: {len(dups)}")
if not dups.empty:
    print(dups.head(20).to_string())

Duplicate (game_id, play_id) pairs: 0


## 5 · Schema diff: 2021 vs 2025

In [7]:
pbp_2021 = nflreadpy.load_pbp([2021]).to_pandas()
print(f"2021 shape: {pbp_2021.shape}")
print(f"2025 shape: {pbp_2025.shape}")

2021 shape: (49922, 372)
2025 shape: (48771, 372)


In [8]:
cols_2021 = set(pbp_2021.columns)
cols_2025 = set(pbp_2025.columns)

added   = sorted(cols_2025 - cols_2021)
dropped = sorted(cols_2021 - cols_2025)
common  = cols_2021 & cols_2025

print(f"\nColumns added in 2025   ({len(added)}):")
for c in added:
    print(f"  + {c}")

print(f"\nColumns dropped from 2021 ({len(dropped)}):")
for c in dropped:
    print(f"  - {c}")

print(f"\nCommon columns: {len(common)}")


Columns added in 2025   (0):

Columns dropped from 2021 (0):

Common columns: 372


In [9]:
# Dtype changes in shared columns
dtype_changes = [
    {'column': c,
     'dtype_2021': str(pbp_2021[c].dtype),
     'dtype_2025': str(pbp_2025[c].dtype)}
    for c in sorted(common)
    if pbp_2021[c].dtype != pbp_2025[c].dtype
]
if dtype_changes:
    print("\n--- Dtype changes in shared columns ---")
    print(pd.DataFrame(dtype_changes).to_string(index=False))
else:
    print("No dtype changes in shared columns.")


--- Dtype changes in shared columns ---
    column dtype_2021 dtype_2025
goal_to_go      int32    float64
